# Qwen3-VL-30B-A3B MoE Expert Activation Analysis (Image vs Text Separated)
## LEGOLite Benchmark with Per-Token Tracking

Designed for Google Colab with NVIDIA G4 Blackwell GPU (96GB VRAM).

This notebook runs the model on LEGOLite and separates expert activations by **token modality** (image vs text).
For each question, it tracks:
- Which experts activate on image tokens
- Which experts activate on text tokens
- Whether they use different experts

**Model architecture:**
- 48 decoder layers, all with MoE blocks
- 128 experts per layer, 8 active per token
- Router: `Qwen3VLMoeTextTopKRouter`

## Setup

In [ ]:
!pip install -q transformers>=4.51.0 accelerate torch pillow pandas tqdm qwen-vl-utils

In [ ]:
import os
import json
import string
import base64
from io import BytesIO
from collections import Counter, defaultdict

import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import Qwen3VLMoeForConditionalGeneration, AutoProcessor

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-30B-A3B-Instruct"
LEGO_TSV_URL = "https://opencompass.openxlab.space/utils/VLMEval/LEGO.tsv"
DATA_DIR = "/content/lego_data"
IMG_DIR = "/content/lego_images"
OUTPUT_DIR = "/content/phase1_image_text_separated"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output will be saved to {OUTPUT_DIR}")

## Load Model

In [ ]:
try:
    import flash_attn
    attn_impl = "flash_attention_2"
except Exception:
    attn_impl = "sdpa"

print(f"Loading {MODEL_ID} (attn: {attn_impl})...")
model = Qwen3VLMoeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
    device_map="auto",
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)

text_config = model.config.text_config
NUM_LAYERS = text_config.num_hidden_layers
NUM_EXPERTS = text_config.num_experts
TOP_K = text_config.num_experts_per_tok

print(f"\nModel loaded successfully.")
print(f"  Layers: {NUM_LAYERS}")
print(f"  Experts: {NUM_EXPERTS}")
print(f"  Top-K: {TOP_K}")

## Hook Infrastructure for Per-Token Tracking

In [ ]:
# expert_log stores per-question+layer data with per-token precision
# expert_log[(q_id, layer_idx)] = {"per_token": defaultdict(Counter), "global": Counter()}
expert_log = defaultdict(lambda: {"per_token": defaultdict(Counter), "global": Counter()})
current_question_id = None
hooks = []

def make_router_hook(layer_idx):
    """
    Hook that captures which experts are selected for each token position.
    Router output is typically (router_logits, router_scores, router_indices).
    router_indices shape: (batch, seq_len, topk) with expert IDs.
    """
    def hook_fn(module, input, output):
        # Extract router_indices robustly
        router_indices = None
        if isinstance(output, (tuple, list)) and len(output) >= 3:
            router_indices = output[2]
        else:
            router_indices = output

        if router_indices is None:
            return

        try:
            ri = router_indices.detach().cpu()
        except Exception:
            return

        # ri might be (batch, seq_len, topk) or (batch, topk)
        if ri.ndim == 3:
            batch, seq_len, topk = ri.shape
            for b in range(batch):
                for t in range(seq_len):
                    vals = ri[b, t]
                    # filter negative/sentinel values
                    experts = [int(x) for x in vals.flatten().tolist() if int(x) >= 0]
                    if experts:
                        expert_log[(current_question_id, layer_idx)]["per_token"][t].update(experts)
                        expert_log[(current_question_id, layer_idx)]["global"].update(experts)
        elif ri.ndim == 2:
            batch, k = ri.shape
            for b in range(batch):
                experts = [int(x) for x in ri[b].flatten().tolist() if int(x) >= 0]
                if experts:
                    expert_log[(current_question_id, layer_idx)]["global"].update(experts)
        else:
            # unknown shape: fallback to global flatten
            indices = [int(x) for x in ri.flatten().tolist() if int(x) >= 0]
            expert_log[(current_question_id, layer_idx)]["global"].update(indices)
    return hook_fn

def find_decoder_layers():
    """Dynamically locate the decoder layer list in the model."""
    for attr_path in ["model.layers", "model.model.layers"]:
        candidate = model
        try:
            for attr in attr_path.split("."):
                candidate = getattr(candidate, attr)
            if hasattr(candidate, '__len__') and len(candidate) == NUM_LAYERS:
                print(f"Found decoder layers at: {attr_path}")
                return candidate
        except AttributeError:
            continue
    for name, module in model.named_modules():
        if hasattr(module, '__len__'):
            try:
                if len(module) == NUM_LAYERS and hasattr(module[0], 'mlp'):
                    print(f"Found decoder layers at: {name}")
                    return module
            except (TypeError, IndexError):
                continue
    raise RuntimeError("Could not find decoder layers.")

decoder_layers = find_decoder_layers()

def register_hooks():
    global hooks
    remove_hooks()
    for layer_idx, layer in enumerate(decoder_layers):
        if hasattr(layer.mlp, 'gate'):
            hook = layer.mlp.gate.register_forward_hook(make_router_hook(layer_idx))
            hooks.append(hook)
    print(f"Registered {len(hooks)} router hooks.")

def remove_hooks():
    global hooks
    for h in hooks:
        try:
            h.remove()
        except Exception:
            pass
    hooks = []

print("Hook infrastructure ready.")

## Load Dataset

In [ ]:
tsv_path = os.path.join(DATA_DIR, "LEGO.tsv")
if not os.path.exists(tsv_path):
    print("Downloading LEGO.tsv (~142MB)...")
    import urllib.request
    urllib.request.urlretrieve(LEGO_TSV_URL, tsv_path)
    print("Download complete.")
else:
    print("LEGO.tsv already downloaded.")

df_full = pd.read_csv(tsv_path, sep="\t")
LITE_CATEGORIES = sorted(df_full["category"].unique())
df_lite = df_full.reset_index(drop=True)
print(f"\nLEGO: {len(df_lite)} questions")
print(df_lite["category"].value_counts().to_string())

full_index_map = {str(row["index"]): row for _, row in df_full.iterrows()}

## Prepare Images

In [ ]:
def save_image_for_row(row):
    """Decode and save base64 image from row."""
    idx = str(row["index"])
    img_col = row.get("image", None)
    if pd.isna(img_col) or img_col is None:
        return []

    img_str = str(img_col)
    if len(img_str) < 20:
        try:
            ref_idx = str(int(float(img_str)))
            if ref_idx in full_index_map:
                img_str = str(full_index_map[ref_idx]["image"])
            else:
                return []
        except (ValueError, TypeError):
            return []

    out_path = os.path.join(IMG_DIR, f"{idx}.png")
    if os.path.exists(out_path):
        return [out_path]

    try:
        img_bytes = base64.b64decode(img_str)
        img = Image.open(BytesIO(img_bytes)).convert("RGB")
        img.save(out_path)
        return [out_path]
    except Exception:
        return []

print("Decoding images...")
image_path_map = {}
for row_idx in tqdm(range(len(df_lite)), desc="Images"):
    row = df_lite.iloc[row_idx]
    idx = str(row["index"])
    image_path_map[idx] = save_image_for_row(row)

decoded = sum(1 for paths in image_path_map.values() if paths)
print(f"Decoded: {decoded} images")

## Build Prompts

In [ ]:
def build_messages(row):
    """Build Qwen3-VL chat messages for a LEGO question."""
    idx = str(row["index"])
    question = row["question"]
    question_type = row["question_type"]
    option_cols = [letter for letter in string.ascii_uppercase if letter in row.index and pd.notna(row[letter])]

    prompt = f"Question: {question}\n"
    if option_cols:
        prompt += "Options:\n"
        for option_letter in option_cols:
            option_value = row[option_letter]
            if isinstance(option_value, str) and "<image" in option_value:
                prompt += f"{option_letter}. [see image]\n"
            else:
                prompt += f"{option_letter}. {option_value}\n"

    if question_type == "sort":
        prompt += "Please respond with only the sequence of letters (e.g., 'BDAC') that correctly orders the steps.\n"
    else:
        prompt += "Please respond with only the letter of the correct answer.\n"

    content = []
    for image_path in image_path_map.get(idx, []):
        content.append({"type": "image", "image": f"file://{image_path}"})
    content.append({"type": "text", "text": prompt})
    return [{"role": "user", "content": content}]

# Test on first question
test_row = df_lite.iloc[0]
test_msgs = build_messages(test_row)
print(f"Test question [{test_row['category']}]: {test_row['question'][:60]}...")
print(f"Images: {len(image_path_map.get(str(test_row['index']), []))}")
print(f"Message items: {len(test_msgs[0]['content'])}")

## Helper: Find Token Positions

In [ ]:
def find_subsequence(full, sub):
    """Find the starting index of subsequence `sub` in list `full`."""
    if not sub or not full:
        return -1
    n, m = len(full), len(sub)
    for i in range(n - m + 1):
        if full[i:i+m] == sub:
            return i
    return -1

print("Helper function ready.")

## Run Inference with Image/Text Separation

In [ ]:
if hasattr(model, 'device') and model.device.type != 'meta':
    input_device = model.device
else:
    input_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Input device: {input_device}")

register_hooks()

results = []
all_expert_data = {}

print(f"\nRunning inference on {len(df_lite)} questions...\n")

for row_idx in tqdm(range(len(df_lite)), desc="Inference"):
    row = df_lite.iloc[row_idx]
    q_id = str(row["index"])
    category = row["category"]
    answer_gt = str(row["answer"])
    current_question_id = q_id

    try:
        messages = build_messages(row)
        
        # Full inputs (image + text)
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(input_device)

        # Text-only inputs for token position alignment
        text_only = [{"role": "user", "content": [{"type": "text", "text": messages[0]["content"][-1]["text"]}]}]
        text_inputs = processor.apply_chat_template(
            text_only,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(input_device)

        # Generate
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=64)

        generated_ids = [full_output[len(full_input):] for full_input, full_output in zip(inputs["input_ids"], output_ids)]
        prediction = processor.batch_decode(
            generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

    except Exception as error:
        prediction = f"ERROR: {error}"
        if row_idx < 3:
            print(f"  Error on q_id={q_id}: {error}")

    # Build token modality mapping
    image_token_positions = set()
    text_token_positions = set()
    try:
        full_ids = inputs["input_ids"][0].detach().cpu().tolist()
        text_ids = text_inputs["input_ids"][0].detach().cpu().tolist()
        start = find_subsequence(full_ids, text_ids)
        if start != -1:
            text_token_positions = set(range(start, start + len(text_ids)))
            image_token_positions = set(range(len(full_ids))) - text_token_positions
        else:
            # Fallback: assume all tokens are text
            text_token_positions = set(range(len(full_ids)))
            image_token_positions = set()
    except Exception:
        # If anything fails, fallback to aggregate-only
        text_token_positions = set()
        image_token_positions = set()

    # Aggregate expert activations
    question_expert_freq_image = Counter()
    question_expert_freq_text = Counter()
    question_expert_freq = Counter()
    per_layer = {}

    for layer_idx in range(NUM_LAYERS):
        key = (q_id, layer_idx)
        if key in expert_log:
            layer_data = expert_log[key]
            per_token = layer_data.get("per_token", {})
            layer_token_map = {}
            for token_pos, c in per_token.items():
                layer_token_map[str(token_pos)] = dict(c)
                if token_pos in text_token_positions:
                    question_expert_freq_text.update(c)
                else:
                    question_expert_freq_image.update(c)
            global_counter = layer_data.get("global", Counter())
            for expert_id, cnt in global_counter.items():
                question_expert_freq.update({expert_id: cnt})
            per_layer[str(layer_idx)] = layer_token_map

    final_image_freq = question_expert_freq_image if sum(question_expert_freq_image.values()) > 0 else question_expert_freq
    final_text_freq = question_expert_freq_text if sum(question_expert_freq_text.values()) > 0 else Counter()

    is_correct = prediction.strip().upper().startswith(answer_gt.strip().upper())

    all_expert_data[q_id] = {
        "question_id": q_id,
        "category": category,
        "prediction": prediction,
        "ground_truth": answer_gt,
        "correct": is_correct,
        "expert_frequency": {str(expert_id): int(count) for expert_id, count in (question_expert_freq.items() or {}).items()},
        "image_expert_frequency": {str(expert_id): int(count) for expert_id, count in final_image_freq.items()},
        "text_expert_frequency": {str(expert_id): int(count) for expert_id, count in final_text_freq.items()},
        "per_layer_experts": per_layer,
    }
    results.append({
        "question_id": q_id,
        "category": category,
        "prediction": prediction,
        "ground_truth": answer_gt,
        "correct": is_correct,
    })

    # Cleanup
    for log_key in [log_key for log_key in list(expert_log.keys()) if log_key[0] == q_id]:
        del expert_log[log_key]
    torch.cuda.empty_cache()

remove_hooks()
print("\nInference complete.")

## Results Summary

In [ ]:
results_df = pd.DataFrame(results)
overall_acc = results_df["correct"].mean()

print("=== LEGOLite Results (Qwen3-VL-30B-A3B) ===")
print()
print(f"Overall accuracy: {overall_acc:.1%}")
print()

for cat in LITE_CATEGORIES:
    cat_df = results_df[results_df["category"] == cat]
    if len(cat_df) > 0:
        acc = cat_df["correct"].mean()
        print(f"  {cat:12s}: {acc:.1%} ({int(cat_df['correct'].sum())}/{len(cat_df)})")

error_count = results_df["prediction"].str.startswith("ERROR:").sum()
if error_count > 0:
    print(f"\n  Errors: {error_count} questions failed during inference")

## Export Results

In [ ]:
# Aggregate by category
category_expert_freq = defaultdict(Counter)
category_image_expert_freq = defaultdict(Counter)
category_text_expert_freq = defaultdict(Counter)

for q_id, question_data in all_expert_data.items():
    cat = question_data["category"]
    
    freq = {int(expert_id): count for expert_id, count in question_data["expert_frequency"].items()}
    category_expert_freq[cat].update(freq)
    
    image_freq = {int(expert_id): count for expert_id, count in question_data["image_expert_frequency"].items()}
    category_image_expert_freq[cat].update(image_freq)
    
    text_freq = {int(expert_id): count for expert_id, count in question_data["text_expert_frequency"].items()}
    category_text_expert_freq[cat].update(text_freq)

# Build summary
category_summary = {}
for cat in LITE_CATEGORIES:
    freq = category_expert_freq[cat]
    image_freq = category_image_expert_freq[cat]
    text_freq = category_text_expert_freq[cat]
    
    if not freq:
        continue
    
    top_experts = freq.most_common(15)
    top_image_experts = image_freq.most_common(15)
    top_text_experts = text_freq.most_common(15)
    
    category_summary[cat] = {
        "top_15_experts": [{"expert_id": expert_id, "activation_count": count} for expert_id, count in top_experts],
        "top_15_image_experts": [{"expert_id": expert_id, "activation_count": count} for expert_id, count in top_image_experts],
        "top_15_text_experts": [{"expert_id": expert_id, "activation_count": count} for expert_id, count in top_text_experts],
        "total_activations": int(sum(freq.values())),
        "total_image_activations": int(sum(image_freq.values())),
        "total_text_activations": int(sum(text_freq.values())),
        "unique_experts_used": int(len(freq)),
        "unique_image_experts": int(len(image_freq)),
        "unique_text_experts": int(len(text_freq)),
    }

print("=== Top Experts by Category (Overall) ===")
print()
for cat in LITE_CATEGORIES:
    info = category_summary.get(cat)
    if not info:
        continue
    print(f"{cat} ({info['total_activations']:,} total, {info['unique_experts_used']} unique):")
    for expert_id, count in info["top_15_experts"][:5]:
        pct = count / info['total_activations'] * 100
        print(f"  Expert {expert_id:3d}: {count:>10,} activations ({pct:.2f}%)")
    print()

## Print Image vs Text Expert Comparison

In [ ]:
print("=== Image vs Text Expert Usage ===")
print()
for cat in LITE_CATEGORIES:
    info = category_summary.get(cat)
    if not info:
        continue
    
    print(f"{cat.upper()}:")
    print(f"  Total activations: {info['total_activations']:,}")
    print(f"    Image: {info['total_image_activations']:,} ({info['total_image_activations']/info['total_activations']*100:.1f}%)")
    print(f"    Text:  {info['total_text_activations']:,} ({info['total_text_activations']/info['total_activations']*100:.1f}%)")
    print()
    print(f"  Unique experts: {info['unique_experts_used']}")
    print(f"    Image only:     {info['unique_image_experts']}")
    print(f"    Text only:      {info['unique_text_experts']}")
    
    # Overlap
    image_set = set(e["expert_id"] for e in info["top_15_image_experts"])
    text_set = set(e["expert_id"] for e in info["top_15_text_experts"])
    overlap = image_set & text_set
    print(f"    Shared (top 15): {len(overlap)} experts")
    print()
    print(f"  Top 5 image experts: {[e['expert_id'] for e in info['top_15_image_experts'][:5]]}")
    print(f"  Top 5 text experts:  {[e['expert_id'] for e in info['top_15_text_experts'][:5]]}")
    print()

## Save JSON

In [ ]:
analysis_report = {
    "model": MODEL_ID,
    "benchmark": "LEGOLite",
    "categories": LITE_CATEGORIES,
    "num_questions": len(df_lite),
    "accuracy": {
        "overall": float(overall_acc),
        **{cat: float(results_df[results_df["category"] == cat]["correct"].mean())
           for cat in LITE_CATEGORIES}
    },
    "category_expert_summary": category_summary,
    "per_question_results": all_expert_data,
}

output_path = os.path.join(OUTPUT_DIR, "lego_moe_expert_analysis_image_text_separated.json")
with open(output_path, "w") as output_file:
    json.dump(analysis_report, output_file, indent=2)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"Results exported to {output_path}")
print(f"File size: {size_mb:.1f} MB")
print(f"\nTo download: use Colab file browser (left panel) or:")
print(f"  from google.colab import files; files.download('{output_path}')")